In [0]:
########################################
# Silent, self-healing state-backfill helpers used by per-lesson setup scripts.
#
# Goal: every demo and lab is runnable standalone. A learner who opens any
# notebook in the course (without having run the earlier ones) still gets a
# correct environment, because each notebook's setup calls the ensure_*
# function for the resources it CONSUMES, and those functions recursively
# build any missing prerequisites back to the raw CSV files.
#
# Design:
#   - The dependency chain is: foundation (CSVs, staged by Classroom-Setup-
#     Common) -> source tables -> orders_bronze -> star schema -> metric view.
#     Each ensure_* calls the ensure_* of its prerequisites, so calling any one
#     resolves the whole chain.
#   - Validation is by ROW COUNT, not mere existence. Because every table is
#     built from the same fixed-seed CSV files, the expected counts are
#     deterministic. A wrong count means a partial or corrupted build, so the
#     helper rebuilds that resource. This catches broken state that an
#     existence-only check would miss.
#   - Rebuilds are surgical: only the resource that is missing/wrong is rebuilt.
#     The one cross-cutting rule is that if any dimension is rebuilt, fact_orders
#     is rebuilt too (its stored surrogate keys would otherwise be stale).
#   - Helpers are silent on success so learners who DID run the earlier lessons
#     see no backfill noise. On a normal in-order run, every ensure_* is just a
#     few fast COUNT checks that short-circuit.
#
# Tolerances (why a few checks are ranges, not strict equality):
#   - dim_customer uses >= 5000 because the optional M09 SCD bonus legitimately
#     MERGEs 20 new customers (5000 -> 5020). We must not wipe that work.
#   - orders_bronze is validated per-file (each required file present with its
#     correct row count) because M06 loads 2024 and M07 legitimately adds 2025.
#     Extra files are fine; missing/short files trigger a clean reload.
########################################

# Resolve the catalog. Common (Classroom-Setup-Common) sets the Python
# variable `my_catalog` from the resolved lab catalog right after USE CATALOG.
# We prefer that, because in some environments (e.g. dbacademy/Vocareum) the
# session's current_catalog() reverts to a default like 'dbacademy' by the time
# this %run executes, which would point every ensure_* at the wrong catalog.
try:
    _catalog = my_catalog  # set by Classroom-Setup-Common
except NameError:
    _catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
_schema      = "vintage_audio"
_volume_path = f"/Volumes/{_catalog}/{_schema}/data/"

# Deterministic expected row counts, fixed by the course CSV files.
_EXPECTED = {
    "customers":   5000,
    "products":     250,
    "stores":        30,
    "dim_customer":5000,   # minimum; the optional M09 SCD bonus may raise to 5020
    "dim_product":  250,
    "dim_store":     30,
    "dim_date":    1461,
    "fact_orders":297998,
}

# Per-file expected row counts for orders_bronze (fixed by the CSVs).
_ORDERS_FILE_COUNTS = {
    "orders_2024_q1.csv": 43188,
    "orders_2024_q2.csv": 47507,
    "orders_2024_q3.csv": 51826,
    "orders_2024_q4.csv": 60463,
    "orders_2025_q1.csv": 51826,
    "orders_2025_q2.csv": 43188,
}

_ckpt_path   = f"/Volumes/{_catalog}/{_schema}/data/_ckpt_orders_bronze"
_schema_path = f"/Volumes/{_catalog}/{_schema}/data/_schema_orders_bronze"


# --------------------------------------------------------------------------
# Low-level helpers
# --------------------------------------------------------------------------
def _table_exists(table: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE `{_catalog}`.`{_schema}`.`{table}`")
        return True
    except Exception:
        return False


def _row_count(table: str) -> int:
    """Row count for a table, or -1 if it does not exist."""
    if not _table_exists(table):
        return -1
    try:
        return spark.sql(
            f"SELECT COUNT(*) AS n FROM `{_catalog}`.`{_schema}`.`{table}`"
        ).collect()[0]["n"]
    except Exception:
        return -1


def _count_ok(table: str, mode: str = "exact") -> bool:
    """Check a table's row count against its deterministic expected value.
    mode='exact' requires equality; mode='min' requires >= expected (used for
    tables a later lesson may legitimately grow, like dim_customer)."""
    expected = _EXPECTED[table]
    actual = _row_count(table)
    if actual < 0:
        return False
    return actual >= expected if mode == "min" else actual == expected


def _drop(table: str) -> None:
    spark.sql(f"DROP TABLE IF EXISTS `{_catalog}`.`{_schema}`.`{table}`")


def _file_rowcount(table: str, filename: str) -> int:
    """How many rows in `table` came from a given source file (0 if none)."""
    if not _table_exists(table):
        return 0
    try:
        return spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM `{_catalog}`.`{_schema}`.`{table}`
            WHERE _metadata.file_name = '{filename}'
        """).collect()[0]["n"]
    except Exception:
        return 0


# --------------------------------------------------------------------------
# Tier 1: source tables (customers, products, stores)
# --------------------------------------------------------------------------
def ensure_source_dimensions() -> None:
    """Ensure customers, products, and stores exist with their expected row
    counts. Rebuilt from the staged CSVs if missing or the wrong size."""
    source_map = {
        "customers_raw": "customers",
        "products_raw":  "products",
        "stores_raw":    "stores",
    }
    for file_stem, table_name in source_map.items():
        if _count_ok(table_name, "exact"):
            continue
        csv_path = f"{_volume_path}{file_stem}.csv"
        spark.sql(f"""
            CREATE OR REPLACE TABLE `{_catalog}`.`{_schema}`.`{table_name}`
            AS SELECT * EXCEPT(_tmp_rescued)
            FROM read_files(
                '{csv_path}',
                format => 'csv',
                header => 'true',
                inferSchema => 'true',
                rescuedDataColumn => '_tmp_rescued'
            )
        """)


# --------------------------------------------------------------------------
# Tier 2: orders_bronze (loaded with Auto Loader)
# --------------------------------------------------------------------------
def _reset_orders_bronze() -> None:
    """Drop orders_bronze and its Auto Loader checkpoint/schema location so a
    clean reload is possible (the checkpoint otherwise skips 'known' files)."""
    _drop("orders_bronze")
    try:
        dbutils.fs.rm(_ckpt_path, recurse=True)
    except Exception:
        pass
    try:
        dbutils.fs.rm(_schema_path, recurse=True)
    except Exception:
        pass


def _autoload_silent(glob_pattern: str) -> None:
    """Run a one-shot Auto Loader batch over files matching glob_pattern,
    appending into orders_bronze. The checkpoint makes this exactly-once."""
    table_name = f"{_catalog}.{_schema}.orders_bronze"
    stream = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", _schema_path)
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("pathGlobFilter", glob_pattern)
            .load(_volume_path)
    )
    query = (
        stream.writeStream
            .option("checkpointLocation", _ckpt_path)
            .trigger(availableNow=True)
            .toTable(table_name)
    )
    query.awaitTermination()


def _orders_files_ok(filenames) -> bool:
    """True if every required order file is present in orders_bronze with its
    correct row count."""
    if not _table_exists("orders_bronze"):
        return False
    return all(_file_rowcount("orders_bronze", f) == _ORDERS_FILE_COUNTS[f]
               for f in filenames)


def ensure_orders_bronze_2024() -> None:
    """Ensure orders_bronze contains the four 2024 quarters, each with its
    correct row count. Extra files (e.g. 2025 added by M07) are left intact."""
    required = ["orders_2024_q1.csv", "orders_2024_q2.csv",
                "orders_2024_q3.csv", "orders_2024_q4.csv"]
    if _orders_files_ok(required):
        return
    # Something is missing or short: rebuild bronze cleanly from the 2024 files.
    _reset_orders_bronze()
    _autoload_silent("orders_2024_q[1234].csv")


def ensure_orders_bronze_full() -> None:
    """Ensure orders_bronze contains all six quarterly files (2024 Q1-Q4 plus
    2025 Q1-Q2), each with its correct row count."""
    required = list(_ORDERS_FILE_COUNTS.keys())
    if _orders_files_ok(required):
        return
    _reset_orders_bronze()
    _autoload_silent("orders_202[45]_q[1234].csv")


# --------------------------------------------------------------------------
# Tier 3: star schema (dim_customer, dim_product, dim_store, dim_date, fact)
# --------------------------------------------------------------------------
def _build_dim_customer() -> None:
    _drop("dim_customer")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_customer (
            customer_key   BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            customer_id    STRING,
            customer_name  STRING,
            loyalty_tier   STRING,
            CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)
        )
    """)
    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_customer
            (customer_id, customer_name, loyalty_tier)
        SELECT customer_id, customer_name, loyalty_tier
        FROM `{_catalog}`.`{_schema}`.customers
        ORDER BY customer_id
    """)


def _build_dim_product() -> None:
    _drop("dim_product")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_product (
            product_key  BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            product_id   STRING,
            product_name STRING,
            category     STRING,
            brand        STRING,
            list_price   DOUBLE,
            CONSTRAINT pk_dim_product PRIMARY KEY (product_key)
        )
    """)
    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_product
            (product_id, product_name, category, brand, list_price)
        SELECT product_id, product_name, category, brand, list_price
        FROM `{_catalog}`.`{_schema}`.products
        ORDER BY product_id
    """)


def _build_dim_store() -> None:
    _drop("dim_store")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_store (
            store_key  BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            store_id   STRING,
            store_name STRING,
            region     STRING,
            state      STRING,
            CONSTRAINT pk_dim_store PRIMARY KEY (store_key)
        )
    """)
    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_store
            (store_id, store_name, region, state)
        SELECT store_id, store_name, region, state
        FROM `{_catalog}`.`{_schema}`.stores
        ORDER BY store_id
    """)


def _build_dim_date() -> None:
    _drop("dim_date")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.dim_date (
            date_key  BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
            full_date DATE,
            year      INT,
            quarter   INT,
            CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
        )
    """)
    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.dim_date
            (full_date, year, quarter)
        SELECT d, YEAR(d), QUARTER(d)
        FROM (
            SELECT DATE_ADD(DATE '2023-01-01', CAST(id AS INT)) AS d
            FROM range(0, 1461)
        )
        ORDER BY d
    """)


def _build_fact_orders() -> None:
    _drop("fact_orders")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.fact_orders (
            order_id       STRING,
            customer_key   BIGINT NOT NULL,
            product_key    BIGINT NOT NULL,
            store_key      BIGINT NOT NULL,
            date_key       BIGINT NOT NULL,
            quantity       INT,
            total_amount   DOUBLE,
            CONSTRAINT fk_fact_orders_customer FOREIGN KEY (customer_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_customer(customer_key) RELY,
            CONSTRAINT fk_fact_orders_product FOREIGN KEY (product_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_product(product_key) RELY,
            CONSTRAINT fk_fact_orders_store FOREIGN KEY (store_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_store(store_key) RELY,
            CONSTRAINT fk_fact_orders_date FOREIGN KEY (date_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_date(date_key) RELY
        )
    """)
    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.fact_orders
            (order_id, customer_key, product_key, store_key, date_key,
             quantity, total_amount)
        SELECT
            ob.order_id, dc.customer_key, dp.product_key, ds.store_key,
            dd.date_key, ob.quantity, ob.total_amount
        FROM `{_catalog}`.`{_schema}`.orders_bronze ob
        JOIN `{_catalog}`.`{_schema}`.dim_customer dc ON ob.customer_id = dc.customer_id
        JOIN `{_catalog}`.`{_schema}`.dim_product  dp ON ob.product_id  = dp.product_id
        JOIN `{_catalog}`.`{_schema}`.dim_store    ds ON ob.store_id    = ds.store_id
        JOIN `{_catalog}`.`{_schema}`.dim_date     dd ON ob.order_date  = dd.full_date
    """)


def ensure_star_schema() -> None:
    """Ensure the full star schema exists and is correctly sized.

    Validates each table by row count and rebuilds only what is missing or
    wrong. If any dimension is rebuilt, fact_orders is rebuilt too, because its
    stored surrogate keys would otherwise point at the old dimension rows.
    Tables a later lesson may legitimately modify are tolerated: dim_customer
    is checked with >= (the SCD bonus can raise it to 5020), and dim_product's
    row count is unaffected by the M09 price_tier column addition.
    """
    ensure_source_dimensions()
    ensure_orders_bronze_full()

    rebuilds = {
        "dim_customer": not _count_ok("dim_customer", "min"),
        "dim_product":  not _count_ok("dim_product",  "exact"),
        "dim_store":    not _count_ok("dim_store",    "exact"),
        "dim_date":     not _count_ok("dim_date",     "exact"),
    }
    any_dim_rebuilt = any(rebuilds.values())

    # If a dimension must be rebuilt, fact_orders depends on it, so drop fact
    # first to avoid dangling references, then rebuild the needed dimensions.
    if any_dim_rebuilt:
        _drop("fact_orders")
    if rebuilds["dim_customer"]:
        _build_dim_customer()
    if rebuilds["dim_product"]:
        _build_dim_product()
    if rebuilds["dim_store"]:
        _build_dim_store()
    if rebuilds["dim_date"]:
        _build_dim_date()

    # Rebuild fact_orders if it is missing/wrong, or if any dimension changed.
    if any_dim_rebuilt or not _count_ok("fact_orders", "exact"):
        _build_fact_orders()


# --------------------------------------------------------------------------
# Tier 4: metric view (vw_product_metrics)
# --------------------------------------------------------------------------
def _view_exists(view: str) -> bool:
    try:
        return spark.sql(
            f"SHOW VIEWS IN `{_catalog}`.`{_schema}` LIKE '{view}'"
        ).count() > 0
    except Exception:
        return False


def ensure_metric_view_product() -> None:
    """Ensure vw_product_metrics exists (built on fact_orders + dim_product).
    Recreated every time the star schema underneath it was rebuilt, and a
    no-op otherwise."""
    ensure_star_schema()
    if _view_exists("vw_product_metrics"):
        return
    spark.sql(f"""
        CREATE OR REPLACE VIEW `{_catalog}`.`{_schema}`.vw_product_metrics
        WITH METRICS LANGUAGE YAML AS $$
        version: 1.1
        source: {_catalog}.{_schema}.fact_orders
        joins:
          - name: dim_product
            source: {_catalog}.{_schema}.dim_product
            "on": dim_product.product_key = source.product_key
        dimensions:
          - name: category
            expr: dim_product.category
            display_name: Category
          - name: brand
            expr: dim_product.brand
            display_name: Brand
        measures:
          - name: total_revenue
            expr: SUM(source.total_amount)
            display_name: Total Revenue
          - name: order_count
            expr: COUNT(source.order_id)
            display_name: Order Count
        $$
    """)
